# Urban Mobility Forecasting

End-to-end Big Data workflow for NYC Yellow Taxi trip records.

## 1. Introduction

This section will present the dataset, source link, project objectives, and target prediction tasks.

## 2. Spark Processing

This section loads, inspects, cleans, transforms, and aggregates the NYC Yellow Taxi trip dataset using both Spark DataFrames and Spark SQL.

### 2.1 Spark Session and Project Paths

Spark is initialized first. The project paths are defined relative to the repository root so the notebook can be run from the `notebooks/` folder without hard-coded absolute paths.

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

spark = (
    SparkSession.builder
    .appName("UrbanMobilityForecasting")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark

### 2.2 Locate Raw Parquet Files

The project expects multiple monthly Yellow Taxi Parquet files in `data/raw/`, for example `yellow_tripdata_2024-01.parquet`, `yellow_tripdata_2024-02.parquet`, and so on. Spark will read all matching files as one distributed dataset.

In [ ]:
parquet_files = sorted(RAW_DATA_DIR.glob("yellow_tripdata_*.parquet"))

print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Number of monthly Parquet files found: {len(parquet_files)}")

for file_path in parquet_files[:12]:
    size_mb = file_path.stat().st_size / (1024 * 1024)
    print(f"- {file_path.name}: {size_mb:.2f} MB")

if not parquet_files:
    raise FileNotFoundError(
        "No Yellow Taxi Parquet files were found in data/raw/. "
        "Download monthly files from the NYC TLC Trip Record Data page before running this section."
    )

### 2.3 Load the Dataset with Spark DataFrames

The Parquet files are loaded into a Spark DataFrame. Parquet is columnar, so Spark can efficiently read only the columns needed for later operations.

In [ ]:
trips_df = spark.read.parquet(*[str(path) for path in parquet_files])

print(f"Number of raw rows: {trips_df.count():,}")
trips_df.printSchema()
trips_df.limit(5).toPandas()

### 2.4 Select Useful Columns

Only the columns needed for analysis and modeling are kept. This makes the next transformations easier to understand and reduces unnecessary data movement.

In [ ]:
selected_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
]

missing_columns = [column for column in selected_columns if column not in trips_df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

trips_selected_df = trips_df.select(*selected_columns)
trips_selected_df.limit(5).toPandas()

### 2.5 Initial Data Quality Checks

Before cleaning, the notebook checks missing values and basic numerical ranges. This helps justify the cleaning rules instead of applying them blindly.

In [ ]:
important_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "payment_type",
    "fare_amount",
    "total_amount",
]

missing_summary_df = trips_selected_df.select([
    F.count(F.when(F.col(column).isNull(), column)).alias(column)
    for column in important_columns
])

missing_summary_df.toPandas()

In [ ]:
numeric_summary_df = trips_selected_df.select(
    F.min("passenger_count").alias("min_passenger_count"),
    F.max("passenger_count").alias("max_passenger_count"),
    F.min("trip_distance").alias("min_trip_distance"),
    F.max("trip_distance").alias("max_trip_distance"),
    F.min("fare_amount").alias("min_fare_amount"),
    F.max("fare_amount").alias("max_fare_amount"),
    F.min("total_amount").alias("min_total_amount"),
    F.max("total_amount").alias("max_total_amount"),
)

numeric_summary_df.toPandas()

### 2.6 Clean and Transform the Data

The following transformations create trip duration and temporal features, then remove records that are invalid or unrealistic for modeling.

In [ ]:
trips_features_df = (
    trips_selected_df
    .withColumn(
        "trip_duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0,
    )
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("is_weekend", F.col("pickup_day_of_week").isin([1, 7]).cast("int"))
    .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
    .withColumn("average_speed_mph", F.col("trip_distance") / (F.col("trip_duration_minutes") / 60.0))
)

clean_trips_df = (
    trips_features_df
    .dropna(subset=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "passenger_count",
        "trip_distance",
        "payment_type",
        "fare_amount",
        "total_amount",
        "trip_duration_minutes",
    ])
    .filter(F.col("trip_duration_minutes") > 0)
    .filter(F.col("trip_duration_minutes") <= 180)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("trip_distance") <= 100)
    .filter(F.col("passenger_count").between(1, 6))
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("fare_amount") <= 500)
    .filter(F.col("total_amount") > 0)
    .filter(F.col("total_amount") <= 1000)
    .filter(F.col("average_speed_mph").between(1, 100))
    .filter(F.col("fare_per_mile").between(1, 100))
    .withColumn(
        "trip_distance_bucket",
        F.when(F.col("trip_distance") < 2, "short")
         .when(F.col("trip_distance") < 8, "medium")
         .otherwise("long"),
    )
)

raw_count = trips_selected_df.count()
clean_count = clean_trips_df.count()

print(f"Raw rows: {raw_count:,}")
print(f"Clean rows: {clean_count:,}")
print(f"Rows removed: {raw_count - clean_count:,}")
print(f"Retention rate: {clean_count / raw_count:.2%}")

### 2.7 Cache the Clean Dataset

The cleaned DataFrame is cached because it is reused by multiple analyses and later machine learning stages.

In [ ]:
clean_trips_df = clean_trips_df.cache()
clean_trips_df.count()
clean_trips_df.limit(5).toPandas()

### 2.8 DataFrame Aggregations

These examples use Spark DataFrame operations for grouped analysis over the cleaned dataset.

In [ ]:
trips_by_hour_df = (
    clean_trips_df
    .groupBy("pickup_hour")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    )
    .orderBy("pickup_hour")
)

trips_by_hour_df.toPandas()

In [ ]:
payment_type_summary_df = (
    clean_trips_df
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip_amount"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    )
    .orderBy(F.desc("trip_count"))
)

payment_type_summary_df.toPandas()

In [ ]:
distance_bucket_summary_df = (
    clean_trips_df
    .groupBy("trip_distance_bucket")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
    )
    .orderBy("trip_distance_bucket")
)

distance_bucket_summary_df.toPandas()

### 2.9 Spark SQL Analysis

The same cleaned DataFrame is registered as a temporary SQL view. This satisfies the Spark SQL requirement and makes the analysis readable for people familiar with SQL.

In [ ]:
clean_trips_df.createOrReplaceTempView("taxi_trips")

In [ ]:
spark.sql("""
    SELECT
        pickup_month,
        COUNT(*) AS trip_count,
        ROUND(SUM(total_amount), 2) AS total_revenue,
        ROUND(AVG(total_amount), 2) AS avg_total_amount,
        ROUND(AVG(trip_distance), 2) AS avg_distance
    FROM taxi_trips
    GROUP BY pickup_month
    ORDER BY pickup_month
""").toPandas()

In [ ]:
spark.sql("""
    SELECT
        is_weekend,
        trip_distance_bucket,
        COUNT(*) AS trip_count,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_minutes
    FROM taxi_trips
    GROUP BY is_weekend, trip_distance_bucket
    ORDER BY is_weekend, trip_distance_bucket
""").toPandas()

### 2.10 Save the Processed Dataset

The cleaned dataset is written to `data/processed/clean_taxi_trips.parquet`. Later sections can load this processed version instead of repeating the full cleaning pipeline.

In [ ]:
processed_output_path = PROCESSED_DATA_DIR / "clean_taxi_trips.parquet"

(
    clean_trips_df
    .write
    .mode("overwrite")
    .parquet(str(processed_output_path))
)

print(f"Processed dataset saved to: {processed_output_path}")

## 3. Spark MLlib Models

This section applies two Spark MLlib methods on the processed taxi dataset. The goal is to show both a regression task and a classification task using distributed machine learning on data produced by the Spark processing stage.

1. **Linear Regression** for predicting `fare_amount`.
2. **Logistic Regression** for classifying trips into `short`, `medium`, or `long` distance categories.

The models are trained and evaluated on the full cleaned 2025 dataset generated in section 2, after removing invalid records and extreme outliers. This keeps the modeling stage aligned with the Big Data objective of the project: the models are not fitted on a small manually selected file, but on the complete processed dataset.


### 3.1 Load the Processed Dataset

The machine learning stage starts from the cleaned dataset created in section 2. The local execution uses the complete processed dataset, not a sample.

Only the columns needed for modeling are selected. Rows with missing modeling values are removed, then the DataFrame is cached because it is reused by both MLlib methods.


In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression

processed_dataset_path = PROCESSED_DATA_DIR / "clean_taxi_trips.parquet"

ml_df = spark.read.parquet(str(processed_dataset_path))
print(f"Processed rows: {ml_df.count():,}")

ml_full_df = (
    ml_df
    .select(
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "trip_duration_minutes",
        "pickup_hour",
        "pickup_day_of_week",
        "pickup_month",
        "is_weekend",
        "PULocationID",
        "DOLocationID",
        "trip_distance_bucket",
    )
    .dropna()
    .cache()
)

print(f"Rows used for MLlib: {ml_full_df.count():,}")


### 3.2 Method 1: Linear Regression for Fare Prediction

The first MLlib method is Linear Regression. The goal is to estimate the taxi fare using numerical trip characteristics such as distance, duration, pickup time, passenger count, and pickup/drop-off zones.

This is a supervised regression problem because the target variable, `fare_amount`, is numerical. The selected features describe both the trip itself and the context in which the trip happened. `VectorAssembler` is used because Spark MLlib expects the input features in a single vector column.

Evaluation metrics: **RMSE**, **MAE**, and **R2**. RMSE and MAE measure the prediction error in fare units, while R2 measures how much of the fare variation is explained by the model.

In [ ]:
regression_features = [
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
    "is_weekend",
    "PULocationID",
    "DOLocationID",
]

regression_assembler = VectorAssembler(
    inputCols=regression_features,
    outputCol="features",
)

regression_ready_df = (
    regression_assembler
    .transform(ml_full_df)
    .select(F.col("fare_amount").alias("label"), "features")
)

regression_train_df, regression_test_df = regression_ready_df.randomSplit([0.8, 0.2], seed=42)

linear_regression = LinearRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=25,
    regParam=0.1,
)

regression_model = linear_regression.fit(regression_train_df)
regression_predictions_df = regression_model.transform(regression_test_df)

rmse = RegressionEvaluator(metricName="rmse").evaluate(regression_predictions_df)
mae = RegressionEvaluator(metricName="mae").evaluate(regression_predictions_df)
r2 = RegressionEvaluator(metricName="r2").evaluate(regression_predictions_df)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2: {r2:.4f}")

regression_predictions_df.select("label", "prediction").show(10, truncate=False)

Result obtained during the local full-dataset run:

- Rows used: `34,921,920`
- RMSE: `5.5359`
- MAE: `2.2669`
- R2: `0.9064`

The R2 value indicates that the model captures a large part of the fare variation using the selected trip features. The MAE value shows that, on average, the prediction error is around 2.27 dollars, which is reasonable for a first linear model trained on large-scale taxi data.


### 3.3 Method 2: Logistic Regression for Trip Category Classification

The second MLlib method is Logistic Regression. The target is `trip_distance_bucket`, which classifies trips as `short`, `medium`, or `long`.

This is a supervised multi-class classification problem. The text labels are converted to numeric labels with `StringIndexer`, and the predictors are again assembled into a single feature vector with `VectorAssembler`.

To avoid a trivial classification problem, `trip_distance` is not used as an input feature in this model. If it were included, the model could almost directly reconstruct the bucket definition instead of learning from related trip characteristics.


In [ ]:
classification_features = [
    "passenger_count",
    "fare_amount",
    "total_amount",
    "trip_duration_minutes",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
    "is_weekend",
    "PULocationID",
    "DOLocationID",
]

label_indexer = StringIndexer(
    inputCol="trip_distance_bucket",
    outputCol="label",
    handleInvalid="skip",
)

indexed_df = label_indexer.fit(ml_full_df).transform(ml_full_df)

classification_assembler = VectorAssembler(
    inputCols=classification_features,
    outputCol="features",
)

classification_ready_df = (
    classification_assembler
    .transform(indexed_df)
    .select("label", "features", "trip_distance_bucket")
)

indexed_df.groupBy("trip_distance_bucket").count().orderBy(F.desc("count")).show(truncate=False)

classification_train_df, classification_test_df = classification_ready_df.randomSplit([0.8, 0.2], seed=42)

logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=30,
    regParam=0.05,
    family="multinomial",
)

classification_model = logistic_regression.fit(classification_train_df)
classification_predictions_df = classification_model.transform(classification_test_df)

accuracy = MulticlassClassificationEvaluator(metricName="accuracy").evaluate(classification_predictions_df)
f1 = MulticlassClassificationEvaluator(metricName="f1").evaluate(classification_predictions_df)
weighted_precision = MulticlassClassificationEvaluator(metricName="weightedPrecision").evaluate(classification_predictions_df)
weighted_recall = MulticlassClassificationEvaluator(metricName="weightedRecall").evaluate(classification_predictions_df)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 score: {f1:.4f}")
print(f"Weighted precision: {weighted_precision:.4f}")
print(f"Weighted recall: {weighted_recall:.4f}")

classification_predictions_df.select("label", "prediction", "trip_distance_bucket").show(10, truncate=False)

In [ ]:
confusion_matrix_df = (
    classification_predictions_df
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

confusion_matrix_df.show(50, truncate=False)

Result obtained during the local full-dataset run:

- Rows used: `34,921,920`
- Accuracy: `0.8078`
- F1 score: `0.7978`
- Weighted precision: `0.8085`
- Weighted recall: `0.8078`

The classification model performs reasonably well even without directly using `trip_distance` as an input feature. Accuracy and weighted recall are close, which means the overall result is stable across the evaluated classes. The F1 score is slightly lower than accuracy, suggesting that some classes are harder to separate, but the model still provides a useful baseline for trip category prediction.


## 4. Data Pipeline

This section defines a Spark ML Pipeline for fare prediction. A pipeline is useful because it groups the feature preparation steps and the model into one reusable workflow.

The pipeline created here has three stages:

1. `VectorAssembler` combines the input columns into one feature vector.
2. `StandardScaler` scales the feature vector.
3. `LinearRegression` trains the fare prediction model.

This satisfies the pipeline requirement and makes the modeling workflow easier to reproduce.

### 4.1 Prepare Pipeline Input Data

The pipeline uses the same cleaned dataset produced in section 2. The target column is `fare_amount`, renamed to `label` because Spark ML estimators expect this convention by default.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StandardScaler, VectorAssembler

pipeline_input_df = spark.read.parquet(str(PROCESSED_DATA_DIR / "clean_taxi_trips.parquet"))

pipeline_features = [
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
    "is_weekend",
    "PULocationID",
    "DOLocationID",
]

pipeline_model_df = (
    pipeline_input_df
    .select(*pipeline_features, F.col("fare_amount").alias("label"))
    .dropna()
    .cache()
)

print(f"Rows used for Spark ML Pipeline: {pipeline_model_df.count():,}")

### 4.2 Build and Train the Spark ML Pipeline

Instead of manually applying each transformation, the stages are declared once and fitted together. When `fit()` is called, Spark learns the scaler parameters and the regression model as part of the same pipeline workflow.

In [ ]:
pipeline_train_df, pipeline_test_df = pipeline_model_df.randomSplit([0.8, 0.2], seed=42)

assembler = VectorAssembler(
    inputCols=pipeline_features,
    outputCol="raw_features",
)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=False,
    withStd=True,
)

pipeline_regression = LinearRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    maxIter=25,
    regParam=0.1,
)

fare_prediction_pipeline = Pipeline(
    stages=[
        assembler,
        scaler,
        pipeline_regression,
    ]
)

pipeline_model = fare_prediction_pipeline.fit(pipeline_train_df)
pipeline_predictions_df = pipeline_model.transform(pipeline_test_df)

pipeline_rmse = RegressionEvaluator(metricName="rmse").evaluate(pipeline_predictions_df)
pipeline_mae = RegressionEvaluator(metricName="mae").evaluate(pipeline_predictions_df)
pipeline_r2 = RegressionEvaluator(metricName="r2").evaluate(pipeline_predictions_df)

print(f"Pipeline RMSE: {pipeline_rmse:.4f}")
print(f"Pipeline MAE: {pipeline_mae:.4f}")
print(f"Pipeline R2: {pipeline_r2:.4f}")

pipeline_predictions_df.select("label", "prediction").show(10, truncate=False)

### 4.3 Pipeline Results

Result obtained during the local full-dataset run:

- Rows used: `34,921,920`
- Pipeline stages: `VectorAssembler`, `StandardScaler`, `LinearRegression`
- RMSE: `5.5427`
- MAE: `2.2685`
- R2: `0.9061`

The results are very close to the Linear Regression model from section 3, which is expected because the prediction target and input features are similar. The difference is that section 4 packages the transformations and model training into a single Spark ML Pipeline, making the workflow easier to reuse and extend.

## 5. UDF and Hyperparameter Tuning

This section adds two required Spark ML elements:

1. a user-defined function (UDF) for creating a custom time-of-day feature;
2. hyperparameter tuning with a parameter grid and `TrainValidationSplit`.

The UDF is used for analysis over the full processed dataset. The tuning step uses Spark ML's native pipeline stages and numeric columns so repeated model training remains scalable.

### 5.1 User-Defined Function

The UDF below converts the numeric pickup hour into a readable period of the day. This kind of transformation is useful when business logic is easier to express as a Python function than as a long expression.

In [ ]:
from pyspark.sql.types import StringType

def pickup_period(hour):
    if hour is None:
        return "unknown"
    if 5 <= hour <= 10:
        return "morning"
    if 11 <= hour <= 15:
        return "midday"
    if 16 <= hour <= 20:
        return "evening_peak"
    return "night"

pickup_period_udf = F.udf(pickup_period, StringType())

udf_input_df = spark.read.parquet(str(PROCESSED_DATA_DIR / "clean_taxi_trips.parquet"))
udf_trips_df = udf_input_df.withColumn("pickup_period", pickup_period_udf(F.col("pickup_hour")))

pickup_period_summary_df = (
    udf_trips_df
    .groupBy("pickup_period")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
    )
    .orderBy(F.desc("trip_count"))
)

pickup_period_summary_df.show(truncate=False)

Result obtained during the local full-dataset run:

| pickup_period | trip_count | avg_fare | avg_distance | avg_duration_minutes |
|---|---:|---:|---:|---:|
| evening_peak | 11,494,318 | 19.21 | 3.27 | 17.11 |
| midday | 10,124,832 | 20.25 | 3.35 | 18.76 |
| night | 7,392,078 | 19.57 | 3.73 | 14.61 |
| morning | 5,910,692 | 19.48 | 3.52 | 16.92 |

The evening peak period has the highest number of trips, while midday trips have the highest average fare and duration.

### 5.2 Hyperparameter Tuning with TrainValidationSplit

The model tuning step uses the fare prediction pipeline idea from section 4 and searches over a small parameter grid for Linear Regression. `TrainValidationSplit` is used instead of a larger cross-validation setup because the dataset is already very large.

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.storagelevel import StorageLevel

tuning_features = [
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
    "is_weekend",
    "PULocationID",
    "DOLocationID",
]

tuning_df = (
    udf_input_df
    .select(*tuning_features, F.col("fare_amount").alias("label"))
    .dropna()
    .persist(StorageLevel.DISK_ONLY)
)

print(f"Rows used for tuning: {tuning_df.count():,}")

tuning_train_df, tuning_test_df = tuning_df.randomSplit([0.8, 0.2], seed=42)

tuning_assembler = VectorAssembler(
    inputCols=tuning_features,
    outputCol="raw_features",
)
tuning_scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=False,
    withStd=True,
)
tuning_regression = LinearRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    maxIter=20,
)

tuning_pipeline = Pipeline(
    stages=[
        tuning_assembler,
        tuning_scaler,
        tuning_regression,
    ]
)

param_grid = (
    ParamGridBuilder()
    .addGrid(tuning_regression.regParam, [0.01, 0.1])
    .addGrid(tuning_regression.elasticNetParam, [0.0, 0.5])
    .build()
)

tuning_evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse",
)

train_validation_split = TrainValidationSplit(
    estimator=tuning_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=tuning_evaluator,
    trainRatio=0.8,
    seed=42,
    parallelism=2,
)

tuning_model = train_validation_split.fit(tuning_train_df)
tuning_predictions_df = tuning_model.transform(tuning_test_df)

tuned_rmse = RegressionEvaluator(metricName="rmse").evaluate(tuning_predictions_df)
tuned_mae = RegressionEvaluator(metricName="mae").evaluate(tuning_predictions_df)
tuned_r2 = RegressionEvaluator(metricName="r2").evaluate(tuning_predictions_df)

best_lr_model = tuning_model.bestModel.stages[-1]
print(f"Best regParam: {best_lr_model.getRegParam()}")
print(f"Best elasticNetParam: {best_lr_model.getElasticNetParam()}")
print(f"Tuned RMSE: {tuned_rmse:.4f}")
print(f"Tuned MAE: {tuned_mae:.4f}")
print(f"Tuned R2: {tuned_r2:.4f}")

tuning_predictions_df.select("label", "prediction").show(10, truncate=False)

### 5.3 Tuning Results

Result obtained during the local full-dataset run:

- Rows used: `34,921,920`
- Tuning method: `TrainValidationSplit`
- Parameter grid: `regParam` in `[0.01, 0.1]`, `elasticNetParam` in `[0.0, 0.5]`
- Best `regParam`: `0.01`
- Best `elasticNetParam`: `0.0`
- Test RMSE: `5.5420`
- Test MAE: `2.2894`
- Test R2: `0.9061`

Validation RMSE by parameter combination:

| regParam | elasticNetParam | validation_rmse |
|---:|---:|---:|
| 0.01 | 0.0 | 5.5264 |
| 0.01 | 0.5 | 5.5264 |
| 0.1 | 0.0 | 5.5276 |
| 0.1 | 0.5 | 5.5290 |

The best model uses the lowest regularization value and no elastic-net mixing. The tuned model remains close to the previous regression results, which suggests that the original Linear Regression setup was already a strong baseline for this feature set.

## 6. TensorFlow Deep Learning

This section trains and evaluates a TensorFlow deep learning model for fare prediction. The target remains `fare_amount`, but the model is now a small feed-forward neural network instead of a Spark ML linear model.

Because the processed dataset contains tens of millions of rows, the TensorFlow script reads the Parquet file in batches using PyArrow instead of loading the full dataset into memory. The model is trained on the full processed dataset with an 80/20 deterministic train/test split.

### 6.1 TensorFlow Model Design

The neural network uses the same main numerical predictors used in the Spark regression models. Features are standardized before training, and the target is also standardized internally so the neural network can learn more efficiently.

Model architecture:

1. `Dense(64, activation="relu")`
2. `Dense(32, activation="relu")`
3. `Dense(1)` output layer for fare prediction

The final evaluation metrics are converted back to fare units, so RMSE and MAE are reported in dollars.

In [ ]:
import json
import subprocess
import sys

tensorflow_script = PROJECT_ROOT / "src" / "run_tensorflow_model.py"

subprocess.run(
    [
        sys.executable,
        str(tensorflow_script),
        "--epochs",
        "2",
        "--batch-size",
        "65536",
    ],
    check=True,
)

### 6.2 Load TensorFlow Results

The script stores the metrics in JSON format so they can be loaded back into the notebook and reported consistently.

In [ ]:
tensorflow_metrics_path = PROCESSED_DATA_DIR / "tensorflow_results" / "tensorflow_metrics.json"

with tensorflow_metrics_path.open("r", encoding="utf-8") as file:
    tensorflow_metrics = json.load(file)

tensorflow_metrics["test_metrics"]

### 6.3 TensorFlow Results

Result obtained during the local full-dataset run:

- Rows used: `34,921,920`
- Training rows: `27,937,536`
- Test rows: `6,984,384`
- Epochs: `2`
- Batch size: `65,536`
- Architecture: `Dense(64, relu)`, `Dense(32, relu)`, `Dense(1)`
- Test RMSE: `5.3210`
- Test MAE: `1.9694`
- Test R2: `0.9135`

The TensorFlow model performs slightly better than the earlier linear regression baseline. This suggests that the neural network is able to capture some non-linear relationships between trip characteristics and fare amount while still using the same cleaned feature set.

## 7. Spark Streaming

This section simulates a streaming workflow with Spark micro-batches. In a production environment this would normally be implemented with Spark Structured Streaming and `writeStream`. On this local Windows setup, native `writeStream` checkpoint metadata requires Hadoop `winutils.exe`, so the project uses a portable Spark micro-batch simulation that applies the same type of streaming transformations and aggregations.

The streaming simulation generates synthetic taxi events, assigns each event to a micro-batch, estimates a fare in real time, and computes streaming analytics by pickup period.

### 7.1 Run the Streaming Simulation

The script creates taxi-like events with Spark, processes them in five micro-batches, and stores the final streaming summary in `data/processed/streaming_results/streaming_summary.json`.

In [ ]:
import json
import subprocess
import sys

streaming_script = PROJECT_ROOT / "src" / "run_streaming_simulation.py"

subprocess.run(
    [
        sys.executable,
        str(streaming_script),
        "--rows-per-second",
        "1000",
        "--duration-seconds",
        "5",
    ],
    check=True,
)

### 7.2 Load Streaming Results

The generated JSON file is loaded back into the notebook so the final report contains reproducible streaming results.

In [ ]:
streaming_metrics_path = PROCESSED_DATA_DIR / "streaming_results" / "streaming_summary.json"

with streaming_metrics_path.open("r", encoding="utf-8") as file:
    streaming_metrics = json.load(file)

streaming_metrics["summary"]

### 7.3 Streaming Results

Result obtained during the local streaming simulation:

- Streaming engine: `Spark micro-batch simulation`
- Source: synthetic taxi events generated with Spark
- Rows per second: `1,000`
- Duration: `5` seconds
- Total events: `5,000`
- Micro-batches: `5`

Per-micro-batch summary:

| micro_batch_id | trip_count | avg_estimated_fare | avg_distance |
|---:|---:|---:|---:|
| 0 | 1,000 | 35.35 | 7.48 |
| 1 | 1,000 | 37.38 | 7.98 |
| 2 | 1,000 | 39.40 | 8.48 |
| 3 | 1,000 | 35.35 | 7.48 |
| 4 | 1,000 | 37.38 | 7.98 |

Final streaming aggregation by pickup period:

| pickup_period | trip_count | avg_estimated_fare | avg_distance | avg_duration_minutes |
|---|---:|---:|---:|---:|
| night | 1,669 | 36.90 | 7.85 | 36.40 |
| morning | 1,251 | 37.25 | 7.97 | 36.53 |
| evening_peak | 1,040 | 37.13 | 7.91 | 36.63 |
| midday | 1,040 | 36.59 | 7.77 | 36.09 |

This completes the end-to-end workflow: large-scale batch processing, Spark SQL analysis, MLlib modeling, pipelines, tuning, TensorFlow deep learning, and streaming-style analytics.